In [1]:
#Aquí importamos todas las paqueterias necesarias para nuestro código

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import roots_legendre
from mpmath import polylog
from scipy.optimize import differential_evolution, root_scalar
from functools import lru_cache

In [6]:
a=[1, 3, 5, 4, 2]
a.sort()
a.reverse()
print(a)

[5, 4, 3, 2, 1]


In [2]:
# VARIABLES
# -----------------------------------------------
# Parametros físicos ajustables

m_init = 92.72
g_init = 1.5
lamda = 0.3
mu = 1
T_v = np.linspace(130, 200, 500)
mu_q_values = np.arange(0, 71, 10)

# -----------------------------------------------
# Parametros ajustables de las integrales por Gauss-Legendre

n2_puntos = 100
n3_puntos = 100
nodos2, pesos2 = roots_legendre(n2_puntos)
nodos3, pesos3 = roots_legendre(n3_puntos)

In [3]:
# Celda en la cual estan contenida la expresion de la segunda derivada del potencial evaluada en phi = 0
# -----------------------------------------------------------------------------

@lru_cache(maxsize=10000)                     # Decorador en Python que utiliza la técnica de memorización
def cached_polylog(z):                       # (caching) para optimizar el rendimiento de funciones
    return float(polylog(2, z))

# -----------------------------------------------------------------------------
#Parte del potencial que corresponde a la Autoenergía

def PI(T_v, mu_q, g, lamda):

    T_v = np.asarray(T_v).item()
    exp_1 = -np.exp(mu_q / T_v)
    exp_2 = -np.exp(-mu_q / T_v)
    term_1 = (lamda * T_v**2) / 2
    t_2 = -((g**2 * T_v**2) / np.pi**2) * (cached_polylog(exp_1) + cached_polylog(exp_2))

    return term_1 + t_2

# -----------------------------------------------------------------------------
#Parte del potencial correspondiente al termino número 3 con término integral mediante Gauss-Legendre

def D2_Int_3(T_v, mu_q, nodos2, pesos2, m, g, lamda):

    T_v = np.asarray(T_v).item()
    k_v = (1 + nodos2) / (1 - nodos2)
    dk = 2 / (1 - nodos2)**2
    Pi = PI(T_v, mu_q, g, lamda)
    arg = k_v**2
    raiz = np.sqrt(arg)
    exp_arg_1 = np.exp(-(-mu_q + raiz) / T_v)
    numerador_1 = exp_arg_1 * g**2
    denominador_1 = 1 + exp_arg_1
    exp_arg_2 = np.exp(-(mu_q + raiz) / T_v)
    numerador_2 = exp_arg_2 * g**2
    denominador_2 = 1 + exp_arg_2

    Int_vals = (16 * T_v * lamda) * k_v**2 * (
        -numerador_1 / (denominador_1 * T_v * raiz)
        -numerador_2 / (denominador_2 * T_v * raiz))

    return np.sum(dk * pesos2 * Int_vals)

# -----------------------------------------------------------------------------
#Parte del potencial correspondiente al termino número 4 con término integral mediante Gauss-Legend

def D2_Int_4(T_v, mu_q, nodos3, pesos3, m, g, lamda):

    T_v = np.asarray(T_v).item()
    k_v = (1 + nodos3) / (1 - nodos3)
    dk = 2 / (1 - nodos3)**2
    Pi = PI(T_v, mu_q, g, lamda)
    arg = -m**2 + k_v**2 + Pi
    raiz = np.sqrt(arg + 0j)
    exp_arg = np.exp(-raiz / T_v)
    numerador = 3 * exp_arg * k_v**2 * lamda
    denominador = 1 - exp_arg

    Int_vals = (8 * T_v * lamda) * (numerador / (denominador * T_v * raiz))

    return np.sum(dk * pesos3 * Int_vals)

# -----------------------------------------------------------------------------
#Función correspondiente que junta y ordena todos los términos anteriores y da el resultado

def Segunda_derivada(T_v, mu_q, lamda, m, g):

    term_1 = 1 / (16 * np.pi**2 * lamda)
    mu_m_ratio = mu**2 / m**2

    if mu_m_ratio <= 0 or mu_m_ratio >= 2:                # Verificar dominio válido
        return np.inf
    term_2 = m**2 * (
        -4 * g**4 - 16 * np.pi**2 * lamda + 9 * lamda**2
        + 3 * lamda**2 * np.log(-mu_m_ratio + 0j)
        - 3 * lamda**2 * np.log(mu_m_ratio / 2)
    )
    term_3 = D2_Int_3(T_v, mu_q, nodos2, pesos2, m, g, lamda)
    term_4 = D2_Int_4(T_v, mu_q, nodos3, pesos3, m, g, lamda)

    return term_1 * (np.real(term_2) - term_3 + term_4)   #Junta todos los términos


In [7]:
# Celda en la cual estan contenida la expresion de la segunda derivada del potencial evaluada en phi = 0
# -----------------------------------------------------------------------------

@lru_cache(maxsize=1000)                     # Decorador en Python que utiliza la técnica de memorización
def cached_polylog(z):                       # (caching) para optimizar el rendimiento de funciones
    return float(polylog(2, z))

# -----------------------------------------------------------------------------
#Parte del potencial que corresponde a la Autoenergía

def PI(T_v, mu_q, g, lamda):

    T_v = np.asarray(T_v).item()
    exp_1 = -np.exp(mu_q / T_v)
    exp_2 = -np.exp(-mu_q / T_v)
    term_1 = (lamda * T_v**2) / 2
    t_2 = -((g**2 * T_v**2) / np.pi**2) * (cached_polylog(exp_1) + cached_polylog(exp_2))

    return term_1 + t_2

# -----------------------------------------------------------------------------
#Parte del potencial correspondiente al termino número 3 con término integral mediante Gauss-Legendre

def D2_Int_3(T_v, mu_q, nodos2, pesos2, m, g, lamda):

    T_v = np.asarray(T_v).item()
    k_v = (1 + nodos2) / (1 - nodos2)
    dk = 2 / (1 - nodos2)**2
    Pi = PI(T_v, mu_q, g, lamda)
    arg = k_v**2
    raiz = np.sqrt(arg)
    exp_arg_1 = np.exp(-(-mu_q + raiz) / T_v)
    numerador_1 = exp_arg_1 * g**2
    denominador_1 = 1 + exp_arg_1
    exp_arg_2 = np.exp(-(mu_q + raiz) / T_v)
    numerador_2 = exp_arg_2 * g**2
    denominador_2 = 1 + exp_arg_2

    Int_vals = (16 * T_v * lamda) * k_v**2 * (
        -numerador_1 / (denominador_1 * T_v * raiz)
        -numerador_2 / (denominador_2 * T_v * raiz))

    return np.sum(dk * pesos2 * Int_vals)

# -----------------------------------------------------------------------------
#Parte del potencial correspondiente al termino número 4 con término integral mediante Gauss-Legend

def D2_Int_4(T_v, mu_q, nodos3, pesos3, m, g, lamda):

    T_v = np.asarray(T_v).item()
    k_v = (1 + nodos3) / (1 - nodos3)
    dk = 2 / (1 - nodos3)**2
    Pi = PI(T_v, mu_q, g, lamda)
    arg = -m**2 + k_v**2 + Pi
    raiz = np.sqrt(arg + 0j)
    exp_arg = np.exp(-raiz / T_v)
    numerador = 3 * exp_arg * k_v**2 * lamda
    denominador = 1 - exp_arg

    Int_vals = (8 * T_v * lamda) * (numerador / (denominador * T_v * raiz))

    return np.sum(dk * pesos3 * Int_vals)

# -----------------------------------------------------------------------------
#Función correspondiente que junta y ordena todos los términos anteriores y da el resultado

def Segunda_derivada(T_v, mu_q, lamda, m, g):

    term_1 = 1 / (16 * np.pi**2 * lamda)
    mu_m_ratio = mu**2 / m**2

    if mu_m_ratio <= 0 or mu_m_ratio >= 2:                # Verificar dominio válido
        return np.inf
    term_2 = m**2 * (
        -4 * g**4 - 16 * np.pi**2 * lamda + 9 * lamda**2
        + 3 * lamda**2 * np.log(-mu_m_ratio + 0j)
        - 3 * lamda**2 * np.log(mu_m_ratio / 2)
    )
    term_3 = D2_Int_3(T_v, mu_q, nodos2, pesos2, m, g, lamda)
    term_4 = D2_Int_4(T_v, mu_q, nodos3, pesos3, m, g, lamda)

    return term_1 * (np.real(term_2) - term_3 + term_4)   #Junta todos los términos


In [8]:
#Celda cuya funcion es encontrar la temperaturas crítica

def find_root(mu_q, T_v, lamda, m, g):
    def f(T):
        return np.real(Segunda_derivada(np.array([T]), mu_q, lamda, m, g))    #Regresa la parte real de las egunda derivada
    try:
        sol = root_scalar(f, bracket=[T_v[0], T_v[-1]], method='brentq')      #Busca la raices entre el primer y ultimo valor de T
        return sol.root if sol.converged else np.nan
    except ValueError:         # Manejar casos donde no hay cambio de signo
        return np.nan


# -----------------------------------------------------------------------------
#Encuentra la raiz para distintos valores de mu_q

def raiz(T_v, lamda, m, g,  mu_q_values):
    respuesta = [find_root(mu_q, T_v, lamda, m, g) for mu_q in mu_q_values] # Aquí se usa una "list comprehension" para evaluar la función find_root
    respuesta = np.array(respuesta)
    if np.isnan(respuesta[0]) or respuesta[0] == 0:   # Verifica si el primer elemento de la respuesta es nan (no es un número)
        return np.full_like(respuesta, np.nan)        # o si es exactamente cero , esto es útil para evitar errores posteriores
    return respuesta                                  # Para indicar que la búsqueda de raíces falló completamente para esa configuración de parámetros.


In [9]:
# Curva de TF de Latice-QCD y su funcion de perdida, además su discretización.

def Tf_QCD():
  Tc = []
  for i in range(len(mu_q_values)):
    x = 3*mu_q_values[i]
    Tc.append(158 - 0.000096*(x**2) + 8.1129e-11*(x**4))# valores multiplicados por TC_0
  Tc = np.array(Tc)
  return Tc

# -----------------------------------------------------------------------------
# Función de perdida que compara los valores obtenidos con los de la temperatura de Lattice-QCD

def loss(parametros, mu_q_values, lamda):
    m, g = parametros
    raiz_valores = raiz(T_v, lamda, m, g, mu_q_values)
    if np.any(np.isnan(raiz_valores)):
        return 1e10
    raiz_QCD = Tf_QCD()
    return np.mean((raiz_valores - raiz_QCD)**2)

In [14]:
# Optimización solo para un valor de lamda
# -----------------------------------------------------------------------------
# Definir los límites para los parámetros m y g

bounds = [(50, 150), (0.5, 5)]

# -----------------------------------------------------------------------------
# Ejecutar la optimización

resultados = differential_evolution(
    loss,
    bounds=bounds,
    args=(mu_q_values, lamda),
    maxiter=100,
    disp=True)

print("Parámetros óptimos encontrados (m, g):", resultados.x)

differential_evolution step 1: f(x)= 0.060229760646694754
differential_evolution step 2: f(x)= 0.060229760646694754
differential_evolution step 3: f(x)= 0.060229760646694754
differential_evolution step 4: f(x)= 0.060229760646694754
differential_evolution step 5: f(x)= 0.060229760646694754
differential_evolution step 6: f(x)= 0.060229760646694754
differential_evolution step 7: f(x)= 0.060229760646694754
differential_evolution step 8: f(x)= 0.060229760646694754
differential_evolution step 9: f(x)= 0.060229760646694754
differential_evolution step 10: f(x)= 0.060229760646694754
differential_evolution step 11: f(x)= 0.060229760646694754
differential_evolution step 12: f(x)= 0.060229760646694754
differential_evolution step 13: f(x)= 0.0573452841954884
differential_evolution step 14: f(x)= 0.0573452841954884
differential_evolution step 15: f(x)= 0.0573452841954884
differential_evolution step 16: f(x)= 0.05724498394192702
differential_evolution step 17: f(x)= 0.05724498394192702
differential_e

In [10]:
# Gráfica de comparación entre Lattice QCD y los parametros calculados
# -----------------------------------------------------------------------------
#Valores obtenidos anteriormente

m_opt, g_opt = resultados.x
Tc_opt = raiz(T_v, lamda, m_opt, g_opt, mu_q_values)
Tc_QCD = Tf_QCD()

# -----------------------------------------------------------------------------
# Graficos obtenidos donde ya pasamos a mu_barionicos

plt.plot(3 * mu_q_values, Tc_QCD, 'o-', label='T_c QCD')        # Pasamos a potencial barionico
plt.plot(3 * mu_q_values, Tc_opt, 'x-', label='T_c Calculada')  # Pasamos a potencial barionico
plt.title('Gráfico de comparación entre Lattice QCD y los parametros calculados')
plt.xlabel('μ_barionico (Mev)')
plt.ylabel('T_c (Mev)')
plt.grid(True)
plt.legend()
plt.show()

NameError: name 'resultados' is not defined

In [11]:
#Muestra la diferencia numerica entre los resultados obtenidos y los de QCD de las temperaturas críticas

Tc_QCD = Tf_QCD()
plt.plot(3 * mu_q_values, np.abs(Tc_QCD-Tc_opt) , 'ro', label='Valores de las diferencias' )    # Pasamos a barionico
plt.title('Diferencia entre los resultados obtenidos y los de QCD de las temperaturas criticas')
plt.xlabel('μ_barionico (Mev)')
plt.ylabel('Tc_QCD - Tc_opt(Mev)')
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'Tc_opt' is not defined

In [15]:
import numpy as np
from scipy.optimize import differential_evolution

def optimizar_para_varios_lambda(lamda_min, lamda_max, paso, num_runs):
    
    # Generar la lista de valores de mu_q (0, 10, ..., 100)
    mu_q_values = np.arange(0, 71, 10)

    # Lista donde se guardarán todos los resultados
    resultados_opt = []

    # Suponiendo que bounds está definido globalmente como en tu código original
    # Por ejemplo: bounds = [(50, 150), (0.5, 5)]
    bounds = [(50, 150), (0.5, 5)]

    # Iterar sobre cada valor de lambda
    for lam in np.arange(lamda_min, lamda_max + 0.001, paso):
        # Para cada lambda, correr la optimización num_runs veces
        for n in range(num_runs):
            res = differential_evolution(
                loss,                  # Función de pérdida definida en tu código
                bounds=bounds,         # Límites para m y g
                args=(mu_q_values, lam),  # Argumentos: mu_q_values y el lambda actual
                maxiter=100,           # Máximo de iteraciones
                disp=False             # Sin mostrar el progreso de cada optimización
            )

            m_opt, g_opt = res.x
            # Agregar la tupla (lam, m_opt, g_opt) a la lista de resultados
            resultados_opt.append((lam, m_opt, g_opt))

        # Imprimir un mensaje al completar las 1000 corridas para este lambda
        print(f"Completadas {num_runs} corridas para λ = {lam:.2f}")

    return resultados_opt


resultados = optimizar_para_varios_lambda(lamda_min=0.4, lamda_max=0.45, paso=0.05, num_runs=100)

C:\Users\limen\AppData\Local\Temp\ipykernel_6696\2955309087.py:60: RuntimeWarning: divide by zero encountered in divide
  Int_vals = (8 * T_v * lamda) * (numerador / (denominador * T_v * raiz))
C:\Users\limen\AppData\Local\Temp\ipykernel_6696\2955309087.py:60: RuntimeWarning: invalid value encountered in divide
  Int_vals = (8 * T_v * lamda) * (numerador / (denominador * T_v * raiz))
C:\Users\limen\AppData\Local\Temp\ipykernel_6696\2955309087.py:60: RuntimeWarning: invalid value encountered in multiply
  Int_vals = (8 * T_v * lamda) * (numerador / (denominador * T_v * raiz))


KeyboardInterrupt: 

In [14]:
import pandas as pd
df = pd.DataFrame(resultados)
df.to_excel("resutados4a45.xlsx", index=False, header=False)  

np.savetxt("mis_resultados4a45.txt", resultados, fmt="%.4f")  


In [6]:
np.arange(0.6, 0.6+ 0.001, 0.05
          )

array([0.6])